# Smart Room: Logistic Regression with Three Chronological Splits
reproducible baseline for predicting room occupancy from sensor readings.

**Target:** `Occupancy`: 0 = unoccupied, 1 = occupied.

| Split | Dates in 2015 (inclusive) | Expected rows | Purpose |
|---|---|---:|---|
| Train | February 2-11 | 11,360 | Learn scaling and model parameters |
| Validation | February 12-15 | 5,760 | Choose regularization strength |
| Test | February 16-18 | 3,440 | Evaluate the selected model |



## 1. Set up the experiment
Put this notebook and the three supplied CSV files in the same folder. Dependencies: Python, pandas, NumPy, scikit-learn, and a Jupyter notebook environment.

Use the same four sensors as the previous notebook for a clear baseline. `HumidityRatio` remains in the original files but is excluded from this baseline, and date is used only to validate the chronological split. No occupancy labels or sensor readings are changed.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, balanced_accuracy_score,
                             confusion_matrix, classification_report, roc_auc_score)

DATA_DIR = Path('.')
FEATURES = ['Temperature', 'Humidity', 'Light', 'CO2']
TARGET = 'Occupancy'
SEED = 42

## 2. Load and verify the three files
Checks verify column structure, counts, dates, numerical validity, binary labels, and chronological separation. They do not verify labels against human observations. Validation and test data are checked for integrity but are not used to learn scaling or coefficients.

In [2]:
specs = {
    'Train': ('Occupancy_Train.csv', '2015-02-02', '2015-02-11', 11360),
    'Validation': ('Occupancy_Validation.csv', '2015-02-12', '2015-02-15', 5760),
    'Test': ('Occupancy_Test.csv', '2015-02-16', '2015-02-18', 3440),
}
expected_columns = ['date', 'Temperature', 'Humidity', 'Light',
                    'CO2', 'HumidityRatio', 'Occupancy']
datasets = {}
split_rows = []
for name, (filename, start, end, expected_rows) in specs.items():
    df = pd.read_csv(DATA_DIR / filename, parse_dates=['date'])
    assert list(df.columns) == expected_columns, f'{name}: unexpected columns'
    assert len(df) == expected_rows, f'{name}: unexpected row count'
    assert not df.isna().any().any(), f'{name}: missing cells'
    assert df['date'].is_monotonic_increasing, f'{name}: timestamps not sorted'
    assert not df['date'].duplicated().any(), f'{name}: repeated timestamps'
    assert df['date'].between(pd.Timestamp(start), pd.Timestamp(end) + pd.Timedelta(days=1), inclusive='left').all()
    assert df[TARGET].isin([0, 1]).all(), f'{name}: invalid target'
    assert set(df[TARGET].unique()) == {0, 1}, f'{name}: only one class'
    assert np.isfinite(df[FEATURES].to_numpy(dtype=float)).all()
    datasets[name] = df
    split_rows.append({'Split': name, 'Rows': len(df),
                       'First timestamp': str(df['date'].min()),
                       'Last timestamp': str(df['date'].max()),
                       'Unoccupied': int(df[TARGET].eq(0).sum()),
                       'Occupied': int(df[TARGET].eq(1).sum()),
                       'Occupied %': round(df[TARGET].mean() * 100, 2)})

assert datasets['Train']['date'].max() < datasets['Validation']['date'].min()
assert datasets['Validation']['date'].max() < datasets['Test']['date'].min()
assert pd.concat(datasets.values())['date'].nunique() == 20560
print(pd.DataFrame(split_rows).to_string(index=False))
print('Verified: chronological splits, no shared timestamps, all 20,560 rows accounted for.')

     Split  Rows     First timestamp      Last timestamp  Unoccupied  Occupied  Occupied %
     Train 11360 2015-02-02 14:19:00 2015-02-11 23:58:59        8445      2915       25.66
Validation  5760 2015-02-12 00:00:00 2015-02-15 23:58:59        5022       738       12.81
      Test  3440 2015-02-16 00:00:00 2015-02-18 09:19:00        2343      1097       31.89
Verified: chronological splits, no shared timestamps, all 20,560 rows accounted for.


## 3. Define features, target, and evaluation metrics
`X` contains the four sensor predictors and `y` the occupancy label. Precision, recall, and F1 refer to the occupied class (1). Accuracy alone can conceal weak detection of the minority occupied class. Balanced accuracy averages recall across both classes. ROC AUC evaluates how well predicted probabilities rank the classes.

In [3]:
X_train, y_train = datasets['Train'][FEATURES], datasets['Train'][TARGET]
X_val, y_val = datasets['Validation'][FEATURES], datasets['Validation'][TARGET]
X_test, y_test = datasets['Test'][FEATURES], datasets['Test'][TARGET]

def evaluate(y_true, predictions, probabilities=None):
    result = {
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(y_true, predictions, zero_division=0),
        'Recall': recall_score(y_true, predictions, zero_division=0),
        'F1': f1_score(y_true, predictions, zero_division=0),
        'Balanced accuracy': balanced_accuracy_score(y_true, predictions),
    }
    if probabilities is not None:
        result['ROC AUC'] = roc_auc_score(y_true, probabilities)
    return result

## 4. Fit logistic regression and choose C on validation data
Logistic regression estimates the probability of occupancy. StandardScaler learns each sensor's mean and standard deviation from **training data only**. The pipeline then applies those same transformations to validation and test inputs.

Compare `C = 0.01, 0.1, 1, 10`. Smaller C means stronger regularization. Select the setting with the highest **validation occupied-class F1**; ties favor smaller C. Keep the classification threshold fixed at 0.5 and class weights unchanged. The test set is not used for selection.

In [4]:
candidate_models = {}
validation_rows = []
for C in [0.01, 0.1, 1.0, 10.0]:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
    ])
    model.fit(X_train, y_train)
    assert model.named_steps['classifier'].n_iter_.max() < 1000, 'Check convergence'
    val_prob = model.predict_proba(X_val)[:, 1]
    val_pred = model.predict(X_val)
    validation_rows.append({'C': C, **evaluate(y_val, val_pred, val_prob)})
    candidate_models[C] = model

validation_results = pd.DataFrame(validation_rows).sort_values(
    ['F1', 'C'], ascending=[False, True]).reset_index(drop=True)
best_C = float(validation_results.loc[0, 'C']) 
best_model = candidate_models[best_C]
print(validation_results.round(4).to_string(index=False))
print(f'Selected C = {best_C:g} using validation F1 only.')

    C  Accuracy  Precision  Recall     F1  Balanced accuracy  ROC AUC
 0.10    0.9915     0.9492  0.9864 0.9674             0.9893   0.9946
10.00    0.9913     0.9503  0.9837 0.9667             0.9881   0.9944
 1.00    0.9911     0.9490  0.9837 0.9661             0.9880   0.9944
 0.01    0.9887     0.9265  0.9905 0.9574             0.9895   0.9960
Selected C = 0.1 using validation F1 only.


## 5. Evaluate the selected model on the test set
Keep the selected model trained on the training split; do not refit on validation data in this baseline. Evaluate the final setting on the test set after selection. Once test results are inspected, do not keep tuning against that set and call it an untouched final test.

In [5]:
test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = best_model.predict(X_test)
test_results = pd.DataFrame([
    {'Model': f'Logistic regression (C={best_C:g})', **evaluate(y_test, test_pred, test_prob)},
])
print(test_results.round(4).to_string(index=False))
print('Classification report:')
report = classification_report(y_test, test_pred, labels=[0, 1],
                               target_names=['Unoccupied', 'Occupied'],
                               digits=4, zero_division=0)
print('\n'.join(line for line in report.splitlines() if 'macro avg' not in line))
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
print('Confusion matrix (rows = actual, columns = predicted):')
print(pd.DataFrame(cm, index=['Actual unoccupied', 'Actual occupied'],
                   columns=['Predicted unoccupied', 'Predicted occupied']).to_string())
tn, fp, fn, tp = cm.ravel()
print(f'Correct unoccupied: {tn}; false occupancy alarms: {fp}; missed occupied: {fn}; correct occupied: {tp}.')

                      Model  Accuracy  Precision  Recall     F1  Balanced accuracy  ROC AUC
Logistic regression (C=0.1)    0.9936     0.9812  0.9991 0.9901             0.9951   0.9979
Classification report:
              precision    recall  f1-score   support

  Unoccupied     0.9996    0.9910    0.9953      2343
    Occupied     0.9812    0.9991    0.9901      1097

    accuracy                         0.9936      3440
weighted avg     0.9937    0.9936    0.9936      3440
Confusion matrix (rows = actual, columns = predicted):
                   Predicted unoccupied  Predicted occupied
Actual unoccupied                  2322                  21
Actual occupied                       1                1096
Correct unoccupied: 2322; false occupancy alarms: 21; missed occupied: 1; correct occupied: 1096.


## 6. Inspect what the model learned
Coefficients describe changes in occupancy log-odds for a one-training-standard-deviation increase in each sensor, holding other predictors constant. They are associations, not causal effects. Correlated sensors can complicate comparisons. The scaler statistics below should match training-only statistics.

In [6]:
scaler = best_model.named_steps['scaler']
classifier = best_model.named_steps['classifier']
assert np.allclose(scaler.mean_, X_train.mean().to_numpy())
coefficients = pd.DataFrame({
    'Sensor': FEATURES,
    'Training mean': scaler.mean_,
    'Training standard deviation': scaler.scale_,
    'Standardized coefficient': classifier.coef_[0], 
})
print(coefficients.round(4).to_string(index=False))
print(f'Intercept: {classifier.intercept_[0]:.4f}')

     Sensor  Training mean  Training standard deviation  Standardized coefficient
Temperature        20.8418                       1.0629                   -0.7043
   Humidity        25.8278                       4.9093                    0.2330
      Light       139.1856                     212.1343                    3.8280
        CO2       634.1499                     307.8090                    1.2565
Intercept: -3.3613


## 7. Review mistakes and explain your experiment
The first ten misclassified test records are shown below. Do not automatically change their labels based on the model: a prediction error does not establish a labeling error.

In [15]:
test_review = datasets['Test'][['date'] + FEATURES + [TARGET]].copy()
test_review['Predicted occupancy'] = test_pred
test_review['Probability occupied'] = test_prob
errors = test_review[test_review[TARGET] != test_review['Predicted occupancy']]
print(f'Misclassified test readings: {len(errors):,} / {len(test_review):,}')
print(errors.head(10).round(4).to_string(index=False))

Misclassified test readings: 22 / 3,440
               date  Temperature  Humidity  Light    CO2  Occupancy  Predicted occupancy  Probability occupied
2015-02-16 08:42:00       20.445    24.340 292.25 831.00          0                    1                0.5979
2015-02-16 08:47:00       20.500    25.050 389.00 856.50          0                    1                0.9041
2015-02-16 08:48:00       20.500    25.025 397.00 870.75          0                    1                0.9202
2015-02-16 08:49:00       20.600    25.050 405.00 869.00          0                    1                0.9253
2015-02-16 08:50:00       20.550    25.100 405.00 872.50          0                    1                0.9287
2015-02-16 08:51:00       20.600    25.200 405.00 871.50          0                    1                0.9265
2015-02-16 11:55:00       21.500    28.175 454.00 967.50          0                    1                0.9663
2015-02-16 11:55:59       21.525    28.175 454.00 967.00          0     